# caps

> What a model takes and what it hands back, without loading it.

Before sending a picture it is worth knowing the model can read one. Nothing here loads weights or touches the network: hosted models are looked up in fastllm's bundled table, and local ones are read off the files already on disk.

::: {.callout-note}
This module is self-contained and has no urai imports. It is a candidate for extraction into its own package once a second consumer wants it — ramabana already reimplements parts of it.
:::

In [ ]:
#| default_exp caps

In [ ]:
#| export
import json
from dataclasses import dataclass
from fastcore.all import Path, first

In [ ]:
#| hide
from fastcore.test import test_eq

## What a model can do

Three tuples rather than a set of booleans, because "takes images" and "makes images" are different questions and a model can do either without the other. `tools` is the third case: a model that generates nothing on its own but will, when handed a server-side generation tool.

`source` records who answered. That matters because `default` means nobody knew, which a caller should treat differently from a table that positively said text-only.

In [ ]:
#| export
MODALITIES = ('text', 'image', 'audio', 'video')   #: in the spelling fastllm's table uses

@dataclass(frozen=True)
class Caps:
    "What a model can be sent, and what it can send back."
    inp: tuple = ('text',)      # modalities it accepts
    out: tuple = ('text',)      # modalities it returns unprompted
    tools: tuple = ()           # modalities it returns when given a server-side generation tool
    source: str = 'default'     # who answered: fastllm|mmproj|config|runtime|fallback|default

    @property
    def gen_image(self): return 'image' in self.out or 'image' in self.tools
    @property
    def gen_video(self): return 'video' in self.out or 'video' in self.tools
    @property
    def known(self):
        "Did anything actually answer? `default` means nobody knew, not that the model is text-only."
        return self.source != 'default'
    def accepts(self, kind): return kind in self.inp

    def fmt(self):
        "One line for a status bar, silent about a plain text-in, text-out model."
        if not self.known: return 'modalities unknown'
        bits = []
        if self.inp != ('text',): bits.append('in: ' + ' '.join(self.inp))
        if self.out != ('text',): bits.append('out: ' + ' '.join(self.out))
        if self.tools: bits.append('via tool: ' + ' '.join(self.tools))
        return ' · '.join(bits)

    def __repr__(self):
        t = f", tools={'+'.join(self.tools)}" if self.tools else ''
        return f"Caps(in={'+'.join(self.inp)}, out={'+'.join(self.out)}{t}, {self.source})"

In [ ]:
c = Caps(('text', 'image'), ('text',), source='fastllm')
test_eq((c.accepts('image'), c.accepts('audio')), (True, False))
test_eq((c.gen_image, c.known), (False, True))
test_eq(c.fmt(), 'in: text image')
test_eq(repr(c), 'Caps(in=text+image, out=text, fastllm)')

In [ ]:
test_eq(Caps().known, False)
test_eq(Caps().fmt(), 'modalities unknown')
test_eq(Caps(source='fastllm').fmt(), '')             # text in, text out: nothing to say
test_eq(Caps(tools=('image',), source='fastllm').gen_image, True)   # only with the tool, but yes

## Hosted models

fastllm ships a table of every model it knows. It is inconsistent about how it says things — some entries list modalities, older ones only set `supports_vision`, and a pure generator says nothing about output at all and only names its `mode`. `tbl_caps` reads all three spellings.

In [ ]:
#| export
#: `mode` values that describe a generator rather than a chat, and what they return.
_gen_modes = {'image_generation': ('image',), 'video_generation': ('video',), 'audio_speech': ('audio',)}

#: Server-side generation tools, by the endpoint that offers them and what they produce.
ENDPOINT_TOOLS = {'/v1/responses': ('image',)}

def tool_modalities(info):
    "What a model generates when handed a server-side tool, which `supported_output_modalities` never says."
    eps = info.get('supported_endpoints') or ()
    return tuple(dict.fromkeys(m for e, ms in ENDPOINT_TOOLS.items() if e in eps for m in ms))

def tbl_caps(model):
    "Modalities from fastllm's bundled model table, or `None` when it says nothing."
    try:
        from fastllm.types import get_model_info
        v, _, m = str(model).partition('/')
        info = get_model_info(m or v, v if m else None) or {}
    except Exception: return None
    inp, out = info.get('supported_modalities'), info.get('supported_output_modalities')
    out = tuple(out) if out else _gen_modes.get(info.get('mode'))
    if not inp: inp = ('text', 'image') if info.get('supports_vision') else None
    if not inp and not out: return None
    return Caps(tuple(inp or ('text',)), tuple(out or ('text',)), tool_modalities(info), 'fastllm')

In [ ]:
c = tbl_caps('gpt-5.1')
test_eq((c.inp, c.out, c.source), (('text', 'image'), ('text', 'image'), 'fastllm'))
test_eq(c.tools, ('image',))              # it serves /v1/responses, so it can be told to draw

In [ ]:
test_eq(tbl_caps('gpt-4o').inp, ('text', 'image'))    # only `supports_vision` was set
test_eq(tbl_caps('gemini-2.5-pro').inp, ('text', 'image', 'audio', 'video'))
test_eq(tbl_caps('dall-e-3').out, ('image',))         # read off `mode`, not the output list
test_eq(tbl_caps('no-such-model-anywhere'), None)

The table lags releases, so a model that shipped last week is simply absent from it. `fallback_caps` covers that by name: every Claude reads images, whether or not the table has heard of this one yet.

In [ ]:
#| export
#: Modalities for vendors fastllm's table leaves blank. Prefix -> (inp, out).
CAPS_FALLBACK = {p: (('text', 'image'), ('text',)) for p in
                 ('claude', 'anthropic/', 'sonnet', 'opus', 'haiku', 'fable')}

def fallback_caps(model):
    "Modalities from `CAPS_FALLBACK`, for the vendors fastllm's table leaves empty."
    s = str(model).lower()
    hit = first(CAPS_FALLBACK.items(), lambda kv: kv[0] in s)
    return Caps(hit[1][0], hit[1][1], (), 'fallback') if hit else None

def hosted_caps(model):
    "Best answer for a hosted model: the table, then the name, then nothing."
    return tbl_caps(model) or fallback_caps(model) or Caps()

In [ ]:
test_eq(fallback_caps('claude-opus-4-1').inp, ('text', 'image'))
test_eq(fallback_caps('gpt-5.1'), None)               # the table already covers it
test_eq(hosted_caps('gpt-5.1').source, 'fastllm')
test_eq(hosted_caps('claude-sonnet-4-5').source, 'fastllm')   # the table knows this one
test_eq(hosted_caps('claude-opus-99').source, 'fallback')     # ...but not this one
test_eq(hosted_caps('no-such-model-anywhere').source, 'default')

The same table knows how big a model's window is, which is the other thing a caller needs before it can decide what fits.

In [ ]:
#| export
DFLT_CTX = 128_000   #: assumed when nothing knows better

def hosted_ctx(model, dflt=DFLT_CTX):
    "Context window for a hosted model, as `(tokens, note)`. The note says so when it was a guess."
    try:
        from fastllm.types import get_model_info
        v, _, m = str(model).partition('/')
        info = get_model_info(m or v, v if m else None) or {}
        if (n := info.get('max_input_tokens') or info.get('max_tokens')): return int(n), ''
    except Exception as e: return dflt, f'context window unknown ({type(e).__name__}), assuming {dflt//1000}k'
    return dflt, f'context window unknown, assuming {dflt//1000}k'

In [ ]:
n, note = hosted_ctx('gpt-5.1')
assert n > 100_000 and note == ''
n, note = hosted_ctx('no-such-model-anywhere')
test_eq((n, note), (DFLT_CTX, 'context window unknown, assuming 128k'))
test_eq(hosted_ctx('no-such-model-anywhere', dflt=8192)[0], 8192)

## Local models

A local model is a path or a hub repo id, and the answer is in the files. A GGUF with an `mmproj` projector beside it reads images; an MLX or transformers repo declares its extra towers in `config.json`.

Both look only at what is already cached. A capability check must not trigger a download.

In [ ]:
#| export
MODEL_EXTS = ('.gguf', '.litertlm')   #: file suffixes that mean "a model, not a repo id"

def is_path(model):
    "Does `model` name a local file rather than a hub repo id?"
    return bool(model) and (str(model).lower().endswith(MODEL_EXTS) or Path(model).exists())

def hub_files(repo):
    "Every already-cached file path for hub repo `repo`. Never touches the network."
    try:
        from huggingface_hub import scan_cache_dir
        r = first(scan_cache_dir().repos, lambda r: r.repo_id == str(repo))
    except Exception: return []
    return [str(f.file_path) for rev in r.revisions for f in rev.files] if r else []

def mmproj_caps(model, model_path=None):
    "A GGUF model with an `mmproj` projector beside it takes images."
    fs = list(hub_files(model)) if model and not is_path(model) else []
    for p in (model_path, model):
        d = Path(p).parent if p else None
        if d and d.is_dir(): fs += [str(x) for x in d.glob('*.gguf')]
    if any('mmproj' in Path(f).name.lower() for f in fs):
        return Caps(('text', 'image'), ('text',), (), 'mmproj')
    return None

In [ ]:
test_eq(is_path('model.gguf'), True)
test_eq(is_path('google/gemma-3-4b-it'), False)
test_eq(is_path(''), False)
test_eq(is_path('.'), True)            # an existing path, whatever it is named

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as d:
    (Path(d)/'qwen3-4b-Q4_K_M.gguf').touch()
    test_eq(mmproj_caps(None, f'{d}/qwen3-4b-Q4_K_M.gguf'), None)     # text-only
    (Path(d)/'mmproj-F16.gguf').touch()
    test_eq(mmproj_caps(None, f'{d}/qwen3-4b-Q4_K_M.gguf').inp, ('text', 'image'))

In [ ]:
#| export
#: `config.json` keys that mark a second tower, by the modality it reads.
_tower_keys = {'image': ('vision_config', 'vision_tower', 'image_token_index'),
               'audio': ('audio_config',)}

def read_cfg(model, model_path=None):
    "A model's `config.json` as a dict, from a local directory or the hub cache, else `{}`."
    for p in (model_path, model):
        if p and (f := Path(p)/'config.json').exists():
            try: return json.loads(f.read_text())
            except Exception: pass
    try:
        from huggingface_hub import hf_hub_download
        return json.loads(Path(hf_hub_download(str(model), 'config.json',
                                               local_files_only=True)).read_text())
    except Exception: return {}

def cfg_caps(model, model_path=None):
    "Modalities a model's `config.json` declares, from the towers it carries beside the text one."
    cfg = read_cfg(model, model_path)
    if not cfg: return None
    inp = ('text',) + tuple(k for k, ks in _tower_keys.items() if any(x in cfg for x in ks))
    return Caps(inp, ('text',), (), 'config')

In [ ]:
with tempfile.TemporaryDirectory() as d:
    (Path(d)/'config.json').write_text('{"model_type": "qwen3"}')
    test_eq(cfg_caps(None, d).inp, ('text',))
    (Path(d)/'config.json').write_text('{"vision_config": {}, "audio_config": {}}')
    test_eq(cfg_caps(None, d).inp, ('text', 'image', 'audio'))

In [ ]:
with tempfile.TemporaryDirectory() as d:
    (Path(d)/'config.json').write_text('not json')
    test_eq(read_cfg(None, d), {})       # unreadable is the same as absent
    test_eq(cfg_caps(None, d), None)
test_eq(cfg_caps(None, '/nonexistent/path'), None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()